In [12]:
%%writefile gemm_benchmark.cu
#include <iostream>
#include <vector>
#include <cmath>
#include <chrono>
#include <cstdlib>
#include <limits>
#include <cuda_runtime.h>

#define CHECK_CUDA(call)                                                      \
    do {                                                                      \
        cudaError_t err = call;                                               \
        if (err != cudaSuccess) {                                             \
            std::cerr << "CUDA Error: " << cudaGetErrorString(err)            \
                      << " at " << __FILE__ << ":" << __LINE__ << std::endl;  \
            exit(EXIT_FAILURE);                                               \
        }                                                                     \
    } while (0)

constexpr int TILE_DIM = 32;

// 1. Naive GEMM Kernel: Global memory access per multiply-accumulate
__global__ void sgemm_naive(const float* __restrict__ A,
                            const float* __restrict__ B,
                            float* __restrict__ C,
                            int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < K; ++k) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

// 2. Shared-Memory Tiled GEMM Kernel: Coalesced memory access and cache reuse
__global__ void sgemm_tiled(const float* __restrict__ A,
                            const float* __restrict__ B,
                            float* __restrict__ C,
                            int M, int N, int K) {
    __shared__ float s_a[TILE_DIM][TILE_DIM];
    __shared__ float s_b[TILE_DIM][TILE_DIM];

    int row = blockIdx.y * TILE_DIM + threadIdx.y;
    int col = blockIdx.x * TILE_DIM + threadIdx.x;

    float sum = 0.0f;
    int numTiles = (K + TILE_DIM - 1) / TILE_DIM;

    for (int t = 0; t < numTiles; ++t) {
        // Load tile from matrix A into shared memory
        int a_col = t * TILE_DIM + threadIdx.x;
        if (row < M && a_col < K) {
            s_a[threadIdx.y][threadIdx.x] = A[row * K + a_col];
        } else {
            s_a[threadIdx.y][threadIdx.x] = 0.0f;
        }

        // Load tile from matrix B into shared memory
        int b_row = t * TILE_DIM + threadIdx.y;
        if (b_row < K && col < N) {
            s_b[threadIdx.y][threadIdx.x] = B[b_row * N + col];
        } else {
            s_b[threadIdx.y][threadIdx.x] = 0.0f;
        }

        // Synchronize threads before computing partial dot-product
        __syncthreads();

        #pragma unroll
        for (int k = 0; k < TILE_DIM; ++k) {
            sum += s_a[threadIdx.y][k] * s_b[k][threadIdx.x];
        }

        // Synchronize before next tile load
        __syncthreads();
    }

    if (row < M && col < N) {
        C[row * N + col] = sum;
    }
}

bool verify_result(const std::vector<float>& h_ref, const float* h_out, int size) {
    constexpr float absolute_tolerance = 1.0e-3f;
    constexpr float relative_tolerance = 1.0e-5f;
    float max_err = 0.0f;
    int mismatch_count = 0;
    for (int i = 0; i < size; ++i) {
        if (!std::isfinite(h_ref[i]) || !std::isfinite(h_out[i])) {
            ++mismatch_count;
            max_err = std::numeric_limits<float>::infinity();
            continue;
        }
        float err = std::abs(h_ref[i] - h_out[i]);
        if (err > max_err) max_err = err;
        float tolerance = absolute_tolerance + relative_tolerance * std::abs(h_ref[i]);
        if (err > tolerance) ++mismatch_count;
    }
    std::cout << "Max absolute error: " << max_err
              << " | Mismatches: " << mismatch_count
              << (mismatch_count == 0 ? " (passed)\n" : " (FAILED)\n");
    return mismatch_count == 0;
}

int main() {
    const int M = 2048;
    const int N = 2048;
    const int K = 2048;
    const size_t bytes_A = static_cast<size_t>(M) * K * sizeof(float);
    const size_t bytes_B = static_cast<size_t>(K) * N * sizeof(float);
    const size_t bytes_C = static_cast<size_t>(M) * N * sizeof(float);

    std::cout << "Matrix Dimensions: " << M << "x" << N << "x" << K << "\n";

    // Host memory allocation
    std::vector<float> h_A(M * K);
    std::vector<float> h_B(K * N);
    std::vector<float> h_C(M * N, 0.0f);
    std::vector<float> h_C_ref(M * N, 0.0f);

    for (int i = 0; i < M * K; ++i) h_A[i] = static_cast<float>(rand()) / RAND_MAX;
    for (int i = 0; i < K * N; ++i) h_B[i] = static_cast<float>(rand()) / RAND_MAX;

    // Device memory allocation
    float *d_A, *d_B, *d_C;
    CHECK_CUDA(cudaMalloc(&d_A, bytes_A));
    CHECK_CUDA(cudaMalloc(&d_B, bytes_B));
    CHECK_CUDA(cudaMalloc(&d_C, bytes_C));

    CHECK_CUDA(cudaMemcpy(d_A, h_A.data(), bytes_A, cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(d_B, h_B.data(), bytes_B, cudaMemcpyHostToDevice));

    dim3 block(TILE_DIM, TILE_DIM);
    dim3 grid((N + TILE_DIM - 1) / TILE_DIM, (M + TILE_DIM - 1) / TILE_DIM);

    cudaEvent_t start, stop;
    CHECK_CUDA(cudaEventCreate(&start));
    CHECK_CUDA(cudaEventCreate(&stop));

    // Warm-up & compute reference via naive kernel
    sgemm_naive<<<grid, block>>>(d_A, d_B, d_C, M, N, K);
    CHECK_CUDA(cudaGetLastError());
    CHECK_CUDA(cudaMemcpy(h_C_ref.data(), d_C, bytes_C, cudaMemcpyDeviceToHost));

    // Benchmark Naive
    CHECK_CUDA(cudaEventRecord(start));
    sgemm_naive<<<grid, block>>>(d_A, d_B, d_C, M, N, K);
    CHECK_CUDA(cudaGetLastError());
    CHECK_CUDA(cudaEventRecord(stop));
    CHECK_CUDA(cudaEventSynchronize(stop));

    float naive_ms = 0.0f;
    CHECK_CUDA(cudaEventElapsedTime(&naive_ms, start, stop));
    double gflops_naive = (2.0 * M * N * K * 1e-9) / (naive_ms * 1e-3);
    std::cout << "[Naive GEMM] Time: " << naive_ms << " ms | GFLOPS: " << gflops_naive << "\n";

    // Warm up the tiled kernel so its first-launch overhead is not timed.
    sgemm_tiled<<<grid, block>>>(d_A, d_B, d_C, M, N, K);
    CHECK_CUDA(cudaGetLastError());
    CHECK_CUDA(cudaDeviceSynchronize());

    // Benchmark Tiled
    CHECK_CUDA(cudaEventRecord(start));
    sgemm_tiled<<<grid, block>>>(d_A, d_B, d_C, M, N, K);
    CHECK_CUDA(cudaGetLastError());
    CHECK_CUDA(cudaEventRecord(stop));
    CHECK_CUDA(cudaEventSynchronize(stop));

    float tiled_ms = 0.0f;
    CHECK_CUDA(cudaEventElapsedTime(&tiled_ms, start, stop));
    double gflops_tiled = (2.0 * M * N * K * 1e-9) / (tiled_ms * 1e-3);
    std::cout << "[Tiled GEMM] Time: " << tiled_ms << " ms | GFLOPS: " << gflops_tiled
              << " | Speedup: " << (naive_ms / tiled_ms) << "x\n";

    // Validate result
    CHECK_CUDA(cudaMemcpy(h_C.data(), d_C, bytes_C, cudaMemcpyDeviceToHost));
    bool result_valid = verify_result(h_C_ref, h_C.data(), M * N);

    // Cleanup
    CHECK_CUDA(cudaFree(d_A));
    CHECK_CUDA(cudaFree(d_B));
    CHECK_CUDA(cudaFree(d_C));
    CHECK_CUDA(cudaEventDestroy(start));
    CHECK_CUDA(cudaEventDestroy(stop));

    return result_valid ? EXIT_SUCCESS : EXIT_FAILURE;
}

Overwriting gemm_benchmark.cu


In [13]:
!nvcc -O3 -arch=sm_75 gemm_benchmark.cu -o gemm_benchmark

In [14]:
!ls -l gemm_benchmark

-rwxr-xr-x 1 root root 1005664 Oct 16 19:36 gemm_benchmark


In [15]:
!./gemm_benchmark

Matrix Dimensions: 2048x2048x2048
[Naive GEMM] Time: 32.2415 ms | GFLOPS: 532.849
[Tiled GEMM] Time: 24.7805 ms | GFLOPS: 693.282 | Speedup: 1.30109x
Max absolute error: 0 | Mismatches: 0 (passed)
